In [1]:
# import necessary libraries
import pandas as pd
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import seaborn as sns
import glob
# set empty values to skyblue

# set png resolution
plt.rcParams['figure.dpi'] = 300

In [2]:
# Compute the Tercile Cutoffs for each unique season and lead category
def get_tercile_cutoffs(df):
    return df.quantile([0.33, 0.66], numeric_only=True)

# Assign the Tercile Category for both predicted_precip and precip
def assign_tercile_category(value, lower_cutoff, upper_cutoff):
    if value <= lower_cutoff:
        return 'Low'
    elif value <= upper_cutoff:
        return 'Medium'
    else:
        return 'High'

In [3]:
# generate a tercile dataframe for one region and one model
def compute_tercile_monthly_df(merged_file_path_csv):

    # extract region and model name from netcdf path
    model = merged_file_path_csv.split('/')[-1].split('_')[-5]
    region = '_'.join(merged_file_path_csv.split('/')[-1].split('_')[0:-5])

    # print model, region
    print(f'Model: {model}')
    print(f'Region: {region}')

    # open netcdf file
    current_df = pd.read_csv(merged_file_path_csv).query('year >= 1993 and year <= 2024')

    # take the spatial means
    current_df = (current_df
                              .groupby(['season', 'lead_category', 'year'])[['predicted_precip', 'precip']]
                              .mean().reset_index())


    # Initialize the `agreement` column
    current_df['agreement'] = np.nan

    # Iterate over each unique combination of month and lead_time, over all years
    for (season, lead_category), group in current_df.groupby(['season', 'lead_category']):
        # Compute tercile cutoffs for predicted_precip and precip for this group (across all years)
        cutoffs = get_tercile_cutoffs(group[['predicted_precip', 'precip']])

        # Extract cutoffs for predicted_precip and precip separately
        lower_cutoff_predicted = cutoffs.loc[0.33, 'predicted_precip']
        upper_cutoff_predicted = cutoffs.loc[0.66, 'predicted_precip']

        lower_cutoff_precip = cutoffs.loc[0.33, 'precip']
        upper_cutoff_precip = cutoffs.loc[0.66, 'precip']

        # assign tercile categories for predicted_precip and precip
        group[f'{model}_tercile_class'] = group['predicted_precip'].apply(assign_tercile_category, args=(lower_cutoff_predicted, upper_cutoff_predicted))
        group['chirps_tercile_class'] = group['precip'].apply(assign_tercile_category, args=(lower_cutoff_precip, upper_cutoff_precip))

        # Calculate agreement: 1 if terciles agree, 0 otherwise
        group['agreement'] = (group[f'{model}_tercile_class'] == group['chirps_tercile_class']).astype(int)

        # Update the DataFrame with the new columns
        current_df.loc[group.index, f'{model}_tercile_class'] = group[f'{model}_tercile_class']
        current_df.loc[group.index, 'chirps_tercile_class'] = group['chirps_tercile_class']
        current_df.loc[group.index, 'agreement'] = group['agreement']
        current_df.loc[group.index, 'model'] = model
        current_df.loc[group.index, 'region'] = region

    # Select the desired columns for the final DataFrame
    final_columns = ['year', 'lead_category', 'season', 'predicted_precip', 'precip', f'{model}_tercile_class', 'chirps_tercile_class', 'agreement', 'model', 'region']
    final_df = current_df[final_columns]
    return final_df

In [4]:
# Initialize an empty dictionary to store DataFrames
dfs_dict = {}

# initiate file list
list_of_files = glob.glob('data/csv/seasonal/seasonal_average/*')

files_path = []

for path in list_of_files:
    path_mod = path.replace('\\', '/')
    files_path.append(path_mod)

# Loop over all files
for f in files_path:
    # Generate the DataFrame
    df = compute_tercile_monthly_df(f)

    # Store the DataFrame in the dictionary with the year as key
    dfs_dict[f] = df

# Concatenate all DataFrames in the dictionary into one DataFrame
final_df = pd.concat(dfs_dict.values(), ignore_index=True)

# Now `final_df` contains all concatenated data

Model: CanESM5
Region: eastern_east_africa
Model: CCSM4
Region: eastern_east_africa
Model: CESM1
Region: eastern_east_africa
Model: CMCC
Region: eastern_east_africa
Model: DWD
Region: eastern_east_africa
Model: ECMWF
Region: eastern_east_africa
Model: GEM5
Region: eastern_east_africa
Model: GFDL
Region: eastern_east_africa
Model: JMA
Region: eastern_east_africa
Model: METEO
Region: eastern_east_africa
Model: MME
Region: eastern_east_africa
Model: NASA
Region: eastern_east_africa
Model: NCEP
Region: eastern_east_africa
Model: CanESM5
Region: eastern_ukraine
Model: CCSM4
Region: eastern_ukraine
Model: CESM1
Region: eastern_ukraine
Model: CMCC
Region: eastern_ukraine
Model: DWD
Region: eastern_ukraine
Model: ECMWF
Region: eastern_ukraine
Model: GEM5
Region: eastern_ukraine
Model: GFDL
Region: eastern_ukraine
Model: JMA
Region: eastern_ukraine
Model: METEO
Region: eastern_ukraine
Model: MME
Region: eastern_ukraine
Model: NASA
Region: eastern_ukraine
Model: NCEP
Region: eastern_ukraine
Mode

In [5]:
# subset the final df by tercile class, low, medium, and high for plotting

# subset by chirps tercile class = low
ss_model_df_low = final_df.query('chirps_tercile_class == "Low"')

# compute agreement rates
ss_model_df_low = ss_model_df_low.groupby(['season', 'lead_category', 'model', 'region'])[['agreement']].mean().reset_index()

# subset by chirps tercile class = medium
ss_model_df_medium = final_df.query('chirps_tercile_class == "Medium"')

# compute agreement rates
ss_model_df_medium = ss_model_df_medium.groupby(['season', 'lead_category', 'model', 'region'])[['agreement']].mean().reset_index()

# subset by chirps tercile class = High
ss_model_df_high = final_df.query('chirps_tercile_class == "High"')

# compute agreement rates
ss_model_df_high = ss_model_df_high.groupby(['season', 'lead_category', 'model', 'region'])[['agreement']].mean().reset_index()

In [6]:
def draw_heatmap(*args, **kwargs):
    data = kwargs.pop('data')
    d = data.pivot(index=args[1], columns=args[0], values=args[2])
    sns.heatmap(d, **kwargs, vmin=0, vmax=1,
                cmap=sns.color_palette('RdYlGn', 10),
                linewidths=0.1, linecolor='black',
                annot=True, fmt=".2f",)
    plt.xticks(fontsize=6)
    plt.yticks(fontsize=6)
    plt.gca().invert_xaxis()

fg = sns.FacetGrid(ss_model_df_low, col='model', row='region', sharex=False, sharey=False)
fg.map_dataframe(draw_heatmap, 'lead_category', 'season', 'agreement', square = True)

fg.set_titles('BN Tercile Hit Rate \n Region={col_name} \n Model={row_name}')
fg.set_ylabels("Season")
fg.set_xlabels("Lead Category")
plt.savefig('figures/bn-tercile_hit_rate-seasonal.png')
plt.close()

In [7]:
def draw_heatmap(*args, **kwargs):
    data = kwargs.pop('data')
    d = data.pivot(index=args[1], columns=args[0], values=args[2])
    sns.heatmap(d, **kwargs, vmin=0, vmax=1,
                cmap=sns.color_palette('RdYlGn', 10),
                linewidths=0.1, linecolor='black',
                annot=True, fmt=".2f",)
    plt.xticks(fontsize=6)
    plt.yticks(fontsize=6)
    plt.gca().invert_xaxis()

fg = sns.FacetGrid(ss_model_df_high, col='model', row='region', sharex=False, sharey=False)
fg.map_dataframe(draw_heatmap, 'lead_category', 'season', 'agreement', square = True)

fg.set_titles('AN Tercile Hit Rate \n Region={col_name} \n Model={row_name}')
fg.set_ylabels("Season")
fg.set_xlabels("Lead Category")
plt.savefig('figures/an-tercile_hit_rate-seasonal.png')
plt.close()

In [8]:
lead_category_order = ['short', 'medium', 'long']

# --- Create the FacetGrid ---
# Adjust height and aspect to control the size of each facet
fg = sns.FacetGrid(
    ss_model_df_low,
    row='region',
    col='model',
    sharey=True,
    sharex=False
)

# --- Map the barplot function to the grid ---
fg.map_dataframe(
    sns.barplot,
    x='season',          # Categories along the x-axis within each facet
    y='agreement',       # Values determining the height of the bars
    hue='lead_category', # Variable to group bars by color
    hue_order=lead_category_order,
    palette='viridis'
)

# --- Add customizations ---
fg.add_legend(title='Lead Category')   # Add a legend for the hue categories
fg.set_axis_labels("Season / Month", "Agreement") # Set overall x and y axis labels
fg.set_titles(row_template="{row_name}", col_template="{col_name}") # Set titles for rows/cols
fg.fig.suptitle('BN Tericle Hit Rate by Region, Model, Season & Lead Category') # Set main title for graph

# Show the plot
plt.savefig('figures/bn-tercile_hit_rate-seasonal-bar.png')
plt.close()

In [9]:
lead_category_order = ['short', 'medium', 'long']

# --- Create the FacetGrid ---
# Adjust height and aspect to control the size of each facet
fg = sns.FacetGrid(
    ss_model_df_high,
    row='region',
    col='model',
    sharey=True,
    sharex=False
)

# --- Map the barplot function to the grid ---
fg.map_dataframe(
    sns.barplot,
    x='season',          # Categories along the x-axis within each facet
    y='agreement',       # Values determining the height of the bars
    hue='lead_category', # Variable to group bars by color
    hue_order=lead_category_order,
    palette='viridis'
)

# --- Add customizations ---
fg.add_legend(title='Lead Category')   # Add a legend for the hue categories
fg.set_axis_labels("Season / Month", "Agreement") # Set overall x and y axis labels
fg.set_titles(row_template="{row_name}", col_template="{col_name}") # Set titles for rows/cols
fg.fig.suptitle('AN Tericle Hit Rate by Region, Model, Season & Lead Category') # Set main title for graph

# Show the plot
plt.savefig('figures/an-tercile_hit_rate-seasonal-bar.png')
plt.close()